[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raulpg14/Quantum-Internet-Network-Simulator/blob/master/notebooks/02_evolution_analysis.ipynb)

# Network evolution analysis — average path length and diameter
Analyses how average shortest path length and diameter scale with number of nodes N at fixed density.
Compatible with local Jupyter and Google Colab.

In [ ]:
import sys
try:
    import qcn
except ImportError:
    import subprocess
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'git+https://github.com/raulpg14/Quantum-Internet-Network-Simulator.git'
    ])
    import qcn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from qcn.engine.simulation import run_simulation
from qcn.engine.config import (
    NETWORK_TYPE_OFBQI, NETWORK_TYPE_SBQI,
    SIM_MODE_EVOLUTION, STYLE_MAP,
    DEFAULT_DENSITY_COEFF_EVOLUTION,
)
matplotlib.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [ ]:
# --- Simulation parameters ---
INITIAL_NODES = 500
DENSITY_COEFF = DEFAULT_DENSITY_COEFF_EVOLUTION
MC_REPS = 5
STEPS = 8
NODE_INCREMENT = 200
SEED = 42
# -----------------------------

In [ ]:
results = {}
for net_type in [NETWORK_TYPE_OFBQI, NETWORK_TYPE_SBQI]:
    res = run_simulation({
        'nodes': INITIAL_NODES,
        'radius': DENSITY_COEFF,
        'type': net_type,
        'mc_iter': MC_REPS,
        'nets_per_mc': STEPS,
        'rad_incr': NODE_INCREMENT,
        'sim_mode': SIM_MODE_EVOLUTION,
        'seed': SEED,
    })
    if res.get('success'):
        results[net_type] = res
        print(f"{net_type}: OK")
    else:
        print(f"{net_type}: ERROR — {res.get('error')}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for net_type, res in results.items():
    style = STYLE_MAP[net_type]
    ax1.plot(res['x_nodes'], res['y_path'],
             color=style['color'], marker='s', linestyle='--',
             markersize=5, label=style['label'])
    ax2.plot(res['x_nodes'], res['y_diameter'],
             color=style['color'], marker='o', linestyle='--',
             markersize=5, label=style['label'])
ax1.set_xlabel('Number of nodes $N$')
ax1.set_ylabel('Average shortest path length $\\langle l \\rangle$')
ax1.set_title('Path length vs N')
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.4)
ax2.set_xlabel('Number of nodes $N$')
ax2.set_ylabel('Diameter')
ax2.set_title('Diameter vs N')
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()
plt.savefig('evolution_analysis.pdf', bbox_inches='tight')
plt.show()
print('Figure saved to evolution_analysis.pdf')

In [ ]:
for net_type, res in results.items():
    fp = res.get('fit_params')
    if fp:
        print(f"{net_type}: {fp['formula']}")
    else:
        print(f"{net_type}: fit not available")